In [2]:
import sympy as sp
from IPython.display import display, Math
import warnings
warnings.filterwarnings("ignore")

# 1. Configuración Matemática
sp.init_printing(use_latex='mathjax')
t, w = sp.symbols('t w', real=True)
j = sp.I  
j_v = sp.Symbol('j') 

def solucionador_ode_fourier_rapido(coeficientes, entrada_f_t_cruda):
    print("==================================================")
    print(" ⚡ SOLUCIONADOR DE ODEs: MODO VELOCIDAD EXTREMA")
    print("==================================================\n")
    
    # ----------------------------------------------------
    # 🛡️ EL PARCHE DE CAUSALIDAD
    # ----------------------------------------------------
    if not entrada_f_t_cruda.has(sp.Heaviside):
        print("🛡️ PARCHE APLICADO: Se inyectó u(t) a la entrada automáticamente.\n")
        entrada_f_t = entrada_f_t_cruda * sp.Heaviside(t)
    else:
        entrada_f_t = entrada_f_t_cruda

    # ----------------------------------------------------
    # ⚙️ CONSTRUCTOR AUTOMÁTICO DE LA ECUACIÓN
    # ----------------------------------------------------
    y = sp.Function('y')(t)
    ecuacion_izq = 0
    grado_maximo = len(coeficientes) - 1
    
    # Armamos la ecuación usando la posición del número en la lista
    for i, coef in enumerate(coeficientes):
        grado_actual = grado_maximo - i
        if grado_actual > 0:
            ecuacion_izq += coef * sp.Derivative(y, t, grado_actual)
        else:
            ecuacion_izq += coef * y
            
    print("1. Ecuación Diferencial en el tiempo (Generada por tu arreglo):")
    display(Math(rf"{sp.latex(ecuacion_izq)} = {sp.latex(entrada_f_t_cruda)} \cdot u(t)"))

    # 2. Transformar Entrada f(t) -> F(w)
    print("\n2. Transformando la entrada f(t) a frecuencia:")
    F_w = sp.simplify(sp.integrate(entrada_f_t * sp.exp(-j*w*t), (t, -sp.oo, sp.oo), conds='none'))
    if isinstance(F_w, sp.Piecewise): F_w = F_w.args[0][0]
    display(Math(rf"V(\omega) = {sp.latex(F_w.subs(j, j_v))}"))

    # 3. Aplicar Fourier a la Izquierda
    print("\n3. Aplicando Fourier a la ecuación diferencial:")
    Y_w_sym = sp.Symbol('I(\omega)')
    
    # Sustitución dinámica de derivadas según el grado
    despeje_fourier = ecuacion_izq
    for grado in range(grado_maximo, 0, -1):
        despeje_fourier = despeje_fourier.subs(sp.Derivative(y, t, grado), (j*w)**grado * Y_w_sym)
    despeje_fourier = despeje_fourier.subs(y, Y_w_sym)
    
    display(Math(rf"{sp.latex(despeje_fourier.subs(j, j_v))} = V(\omega)"))

    # 4. Obtener Función de Transferencia G(w)
    print("\n4. Despejando la Función de Transferencia G(w):")
    polinomio_caracteristico = sp.simplify(despeje_fourier / Y_w_sym)
    G_w = 1 / polinomio_caracteristico
    display(Math(rf"G(\omega) = \frac{{1}}{{{sp.latex(polinomio_caracteristico.subs(j, j_v))}}}"))

    # 5. Respuesta en Frecuencia Total
    print("\n5. Respuesta Total en Frecuencia I(w) = G(w) * V(w):")
    I_w = sp.simplify(G_w * F_w)
    display(Math(rf"I(\omega) = {sp.latex(I_w.subs(j, j_v))}"))

    # 6. Fracciones Parciales
    print("\n6. Desarrollo por Fracciones Parciales:")
    I_w_partida = sp.apart(I_w, w)
    display(Math(rf"I(\omega) = {sp.latex(I_w_partida.subs(j, j_v))}"))

    print("\n7. RESULTADO FINAL (Solución Particular i(t)):")
    s = sp.symbols('s')
    I_s = I_w.subs(j*w, s).subs(w, s/j)
    i_t = sp.inverse_laplace_transform(I_s, s, t).subs(sp.Heaviside(t), 1)
    
    display(Math(rf"i(t) = \left[ {sp.latex(sp.simplify(i_t))} \right] u(t)"))
    print("==================================================")

# ==========================================
# 📝 ZONA DE CONFIGURACIÓN RÁPIDA (TU EXAMEN)
# ==========================================
# Ingresa SOLO los coeficientes de izquierda a derecha
# Ej: y'' + 4y' + 3y ---> [1, 4, 3]
mis_coeficientes = [1, 4, 3]

# Lado derecho: La función cruda del examen
entrada = sp.exp(-t)

solucionador_ode_fourier_rapido(mis_coeficientes, entrada)

 ⚡ SOLUCIONADOR DE ODEs: MODO VELOCIDAD EXTREMA

🛡️ PARCHE APLICADO: Se inyectó u(t) a la entrada automáticamente.

1. Ecuación Diferencial en el tiempo (Generada por tu arreglo):


<IPython.core.display.Math object>


2. Transformando la entrada f(t) a frecuencia:


<IPython.core.display.Math object>


3. Aplicando Fourier a la ecuación diferencial:


...

<IPython.core.display.Math object>


6. Desarrollo por Fracciones Parciales:


<IPython.core.display.Math object>


7. RESULTADO FINAL (Solución Particular i(t)):


<IPython.core.display.Math object>